# Wan 2.2 Kaggle: handoff-frame policy test

This notebook tests which handoff frame is useful for sequential shots. It generates one release clip, extracts three candidate handoff frames, displays them as a contact sheet, and then runs continuation clips from the selected candidate. The comparison is about handoff timing, not model quality or GPU placement.


In [ ]:
import os, subprocess, json, shutil
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ROOT = Path("/kaggle/working")
IMAGE = Path("/kaggle/input/datasets/abhinavk0006/testimage/1.jpeg")
REPOS = {
    "Wan2.2": ("https://github.com/abhinavk0006/Wan2.2.git", "78fb5ce"),
    "image-to-video-pipeline": ("https://github.com/abhinavk0006/image-to-video-pipeline.git", "07c7f2f"),
    "Edu-video-gen-dataset": ("https://github.com/abhinavk0006/Edu-video-gen-dataset.git", "02b327b"),
}
for name, (url, commit) in REPOS.items():
    path = ROOT / name
    if not path.exists(): subprocess.run(["git", "clone", url, str(path)], check=True)
    subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(path), "checkout", "--detach", commit], check=True)
assert IMAGE.is_file(), IMAGE
print("Pinned:", REPOS)


In [ ]:
%pip install -q -r /kaggle/working/Edu-video-gen-dataset/requirements.txt imageio-ffmpeg
%pip install -q 'diffusers>=0.31,<0.33' 'transformers>=4.49,<=4.51.3' 'accelerate>=1.1.1' easydict safetensors ftfy huggingface_hub imageio


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 2), "GiB")
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2


## Generate the release clip

The release clip is 2 seconds / 33 frames. It should release one visible droplet and hold the scene long enough to inspect candidate handoff frames.


In [ ]:
import json
pipeline = ROOT / "image-to-video-pipeline"
fixture_dir = pipeline / "kaggle_experiments"
fixture_dir.mkdir(exist_ok=True)
release_fixture = {
  "name": "Handoff Release Clip",
  "clips": [{
    "name": "release_one_drop",
    "scene": {"scene": "Macro test tube held by a clamp with a pipette above white crystals.", "camera": "Locked macro shot", "lighting": "Soft neutral laboratory lighting"},
    "prompt_bundle": {
      "motion_prompt": "Animate the reference apparatus. The pipette releases one transparent droplet, then stops completely. Hold the single visible droplet in mid-air near the end of the shot. Do not create any additional liquid.",
      "negative_prompt": "multiple droplets, second droplet, liquid stream, continuous dripping, pouring, splash, extra pipette, duplicated apparatus, camera shake, zoom, cuts, text, labels",
      "clip_duration_seconds": 2.0
    }
  }]
}
release_path = fixture_dir / "handoff_release_clip.json"
release_path.write_text(json.dumps(release_fixture, indent=2), encoding="utf-8")
print(release_path)


In [ ]:
%cd /kaggle/working/image-to-video-pipeline
!python main.py --knowledge-file /kaggle/working/image-to-video-pipeline/kaggle_experiments/handoff_release_clip.json --initial-image /kaggle/input/datasets/abhinavk0006/testimage/1.jpeg --output-dir /kaggle/working/handoff_release_output --generator-script kaggle_wan_wrapper.py --generator-working-dir /kaggle/working/image-to-video-pipeline --wan-repo-dir /kaggle/working/Wan2.2 --wan-model-preset i2v-a14b --continuity-mode independent --generator-arg=--lightning --generator-arg=--gpu0-memory-gib=6 --generator-arg=--gpu1-memory-gib=10 --generator-arg=--memory-telemetry


## Extract and inspect candidate handoff frames

For a 2-second clip, compare:
- `final`: approximately 0.05 seconds before the end;
- `near_end`: 0.5 seconds before the end;
- `mid`: 1.0 second before the end.

Choose the latest candidate where the object required by the next prompt is visibly present and not duplicated. This is the only human decision in this calibration notebook.


In [ ]:
from imageio_ffmpeg import get_ffmpeg_exe
from PIL import Image, ImageDraw
import subprocess, math

release_video = Path("/kaggle/working/handoff_release_output/handoff_release_clip/release_one_drop.mp4")
candidate_dir = release_video.parent / "handoff_candidates"
candidate_dir.mkdir(exist_ok=True)
ffmpeg = get_ffmpeg_exe()
# duration is known from the fixture; seek times are 1.95s, 1.50s, and 1.00s.
candidates = {"final": 1.95, "near_end": 1.50, "mid": 1.00}
paths = []
for name, timestamp in candidates.items():
    output = candidate_dir / f"{name}.png"
    subprocess.run([ffmpeg, "-y", "-ss", str(timestamp), "-i", str(release_video), "-frames:v", "1", str(output)], check=True, capture_output=True)
    paths.append((name, output))
images = [Image.open(path).convert("RGB") for _, path in paths]
w = min(image.width for image in images)
images = [image.resize((w, int(image.height * w / image.width))) for image in images]
label_h = 32
sheet = Image.new("RGB", (w * len(images), images[0].height + label_h), "white")
draw = ImageDraw.Draw(sheet)
for index, ((name, _), image) in enumerate(zip(paths, images)):
    x = index * w
    sheet.paste(image, (x, label_h))
    draw.text((x + 8, 8), name, fill="black")
sheet_path = candidate_dir / "handoff_candidates.jpg"
sheet.save(sheet_path, quality=95)
display(sheet)
print("Candidates:", paths)
print("Select one name in the next cell: final, near_end, or mid")


In [ ]:
# Set this after inspecting the contact sheet.
SELECTED_HANDOFF = "mid"  # change to: final, near_end, or mid
assert SELECTED_HANDOFF in {"final", "near_end", "mid"}
selected_frame = Path("/kaggle/working/handoff_release_output/handoff_release_clip/handoff_candidates") / f"{SELECTED_HANDOFF}.png"
assert selected_frame.is_file(), selected_frame
print("Selected handoff:", selected_frame)


## Generate the continuation from the selected handoff

This is deliberately independent of the release clip's extracted frame logic: it uses the selected candidate image explicitly, so you can compare handoff policies without regenerating the release clip.


In [ ]:
continuation_fixture = {
  "name": f"Handoff Continuation {SELECTED_HANDOFF}",
  "clips": [{
    "name": "continue_visible_drop",
    "input_frame_path": str(selected_frame),
    "scene": {"scene": "The same test-tube composition with the one droplet visible.", "camera": "Locked macro shot", "lighting": "Soft neutral laboratory lighting"},
    "prompt_bundle": {
      "motion_prompt": "Animate the single transparent droplet already visible in the reference image. Move that same droplet slowly downward toward the crystals. The pipette is closed and inactive. Do not create, emit, or add any liquid.",
      "negative_prompt": "new droplet, second droplet, multiple droplets, liquid stream, continuous dripping, pouring, splash, extra pipette, duplicated apparatus, camera shake, zoom, cuts, text, labels",
      "clip_duration_seconds": 2.0
    }
  }]
}
continuation_path = fixture_dir / f"handoff_continuation_{SELECTED_HANDOFF}.json"
continuation_path.write_text(json.dumps(continuation_fixture, indent=2), encoding="utf-8")
print(continuation_path)


In [ ]:
%cd /kaggle/working/image-to-video-pipeline
!python main.py --knowledge-file /kaggle/working/image-to-video-pipeline/kaggle_experiments/handoff_continuation_{SELECTED_HANDOFF}.json --output-dir /kaggle/working/handoff_continuation_{SELECTED_HANDOFF} --generator-script kaggle_wan_wrapper.py --generator-working-dir /kaggle/working/image-to-video-pipeline --wan-repo-dir /kaggle/working/Wan2.2 --wan-model-preset i2v-a14b --continuity-mode independent --generator-arg=--lightning --generator-arg=--gpu0-memory-gib=6 --generator-arg=--gpu1-memory-gib=10 --generator-arg=--memory-telemetry


## Interpretation

A useful handoff frame should visibly contain the entity named by the continuation prompt. If `mid` contains the droplet but `final` does not, use an entity-aware earlier handoff. If none contains the correct state, do not chain: generate an independent shot from a verified keyframe or composite the droplet separately.
